# Financial Sentiment Dataset Builder

This notebook downloads, cleans, and merges news and social media datasets to create a comprehensive master dataset. This data is intended for Natural Language Processing (NLP) tasks, such as fine-tuning models like FinBERT.

**Key Features:**
* Uses a pure CSV mirror for the news data to prevent Hugging Face script errors.
* Combines Financial PhraseBank (News) and Twitter Financial News (Tweets).
* Standardizes labels (positive, negative, neutral) and deduplicates the final dataset.

In [1]:
# Install required libraries for data manipulation and dataset loading
!pip install datasets pandas numpy

import pandas as pd
from datasets import load_dataset


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\david.suarez\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
C:\Users\david.suarez\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Data Ingestion Functions

In this section, we define the functions to fetch and preprocess our two distinct data sources:
1. **Financial News**: Sourced from a CSV mirror of the Financial PhraseBank.
2. **Financial Tweets**: Sourced from the Twitter Financial News Sentiment dataset.

Both functions will standardize the column names to `text`, `sentiment`, and `source` to make merging seamless.

In [2]:
def load_and_prep_phrasebank():
    """
    Loads the Financial PhraseBank dataset (News).
    Uses a pure CSV mirror ('chiapudding/kaggle-financial-sentiment') 
    to guarantee no Hugging Face script execution errors.
    """
    print("Loading Financial PhraseBank (News) via pure CSV mirror...")
    
    # Load the training split of the dataset
    dataset = load_dataset("chiapudding/kaggle-financial-sentiment")
    df_news = pd.DataFrame(dataset['train'])
    
    # Dynamically rename columns to a standard format (text, sentiment)
    col_mapping = {}
    for col in df_news.columns:
        if col.lower() in ['sentence', 'text']:
            col_mapping[col] = 'text'
        elif col.lower() in ['sentiment', 'label']:
            col_mapping[col] = 'sentiment'
            
    df_news = df_news.rename(columns=col_mapping)
    
    # Ensure sentiment is string-based and lowercase
    if pd.api.types.is_numeric_dtype(df_news['sentiment']):
        label_mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}
        df_news['sentiment'] = df_news['sentiment'].map(label_mapping)
    else:
        df_news['sentiment'] = df_news['sentiment'].astype(str).str.lower()
        
    # Add a source column for tracking
    df_news['source'] = 'news'
    
    # Return only the relevant columns
    return df_news[['text', 'sentiment', 'source']]

In [3]:
def load_and_prep_tweets():
    """
    Loads the Twitter Financial News Sentiment dataset.
    This dataset is natively safe and script-free.
    """
    print("Loading Twitter Financial News Dataset (Tweets)...")
    dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")
    
    # Convert both train and validation splits to DataFrames
    df_train = pd.DataFrame(dataset['train'])
    df_val = pd.DataFrame(dataset['validation'])
    
    # Combine the splits into a single DataFrame
    df_tweets = pd.concat([df_train, df_val], ignore_index=True)
    
    # Map numeric labels to standard string categories
    label_mapping = {0: 'negative', 1: 'positive', 2: 'neutral'}
    df_tweets['sentiment'] = df_tweets['label'].map(label_mapping)
    
    # Add a source column for tracking
    df_tweets['source'] = 'twitter'
    
    # Return only the relevant columns
    return df_tweets[['text', 'sentiment', 'source']]

### Merging and Cleaning Pipeline

This function acts as the orchestrator. It calls our ingestion functions, merges the outputs, and performs necessary cleaning steps such as removing duplicates, dropping missing values, and shuffling the final dataset.

In [4]:
def build_master_dataset():
    """
    Executes the pipeline to fetch, merge, and clean the definitive dataset.
    """
    # 1. Load both datasets
    df_news = load_and_prep_phrasebank()
    df_tweets = load_and_prep_tweets()
    
    # 2. Merge them together
    print("Merging datasets...")
    df_master = pd.concat([df_news, df_tweets], ignore_index=True)
    
    # 3. Clean the combined data
    print("Cleaning data...")
    # Remove exact duplicate texts to prevent data leakage
    df_master = df_master.drop_duplicates(subset=['text'])
    # Drop any rows with missing values
    df_master = df_master.dropna()
    # Shuffle the dataset (frac=1) and reset the index
    df_master = df_master.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # 4. Print summary statistics
    print("\n--- Master Dataset Statistics ---")
    print(f"Total samples: {len(df_master)}")
    print("\nBreakdown by Source:")
    print(df_master['source'].value_counts())
    print("\nBreakdown by Sentiment:")
    print(df_master['sentiment'].value_counts())
    
    return df_master

### Execution and Export

Finally, we trigger the pipeline and save our prepared dataset to a CSV file. This file will be ready for the next phase of our NLP project (e.g., tokenization and model training).

In [ ]:
# Execute the pipeline to build the master DataFrame
master_dataset = build_master_dataset()

# Define the output path
output_filename = "../data/sentiment_dataset.csv"

# Save the definitive dataset to a CSV file without the index column
master_dataset.to_csv(output_filename, index=False)

print(f"\nDataset successfully saved to {output_filename}")

Loading Financial PhraseBank (News) via pure CSV mirror...
Loading Twitter Financial News Dataset (Tweets)...
Merging datasets...
Cleaning data...

--- Master Dataset Statistics ---
Total samples: 16258

Breakdown by Source:
source
twitter    11931
news        4327
Name: count, dtype: int64

Breakdown by Sentiment:
sentiment
neutral     10082
positive     3878
negative     2298
Name: count, dtype: int64

Dataset successfully saved to data/sentiment_dataset.csv


### Expanding the Dataset with Additional Financial News Sources

The initial master dataset is heavily imbalanced (≈62% neutral, 24% positive, 14% negative), which is suboptimal for fine-tuning techniques like **rsLoRA** that benefit from balanced class distributions to learn discriminative features for minority classes.

To address this, we incorporate two additional high-quality financial NLP datasets:
1. **FinGPT Sentiment Train** (`FinGPT/fingpt-sentiment-train`): A large-scale financial sentiment corpus with rich negative/positive examples.
2. **Financial Classification** (`nickmuchi/financial-classification`): Curated financial news with balanced sentiment annotations.

Both will be normalized into our standard `text`, `sentiment`, `source` schema before merging.

In [23]:
def load_and_prep_fingpt():
    """
    Loads the FinGPT sentiment training dataset.
    Maps its 5-class scheme (strong negative/negative/neutral/positive/strong positive)
    into our 3-class standard (negative, neutral, positive).
    """
    print("Loading FinGPT Sentiment Dataset (Financial News)...")
    dataset = load_dataset("FinGPT/fingpt-sentiment-train")
    df_fingpt = pd.DataFrame(dataset['train'])

    # The dataset uses 'input' for text and 'output' for sentiment label
    df_fingpt = df_fingpt.rename(columns={'input': 'text', 'output': 'sentiment'})

    # Normalize 5-class labels into our 3-class standard
    label_mapping = {
        'strong negative': 'negative',
        'moderately negative': 'negative',
        'mildly negative': 'negative',
        'negative': 'negative',
        'neutral': 'neutral',
        'mildly positive': 'positive',
        'moderately positive': 'positive',
        'strong positive': 'positive',
        'positive': 'positive'
    }
    df_fingpt['sentiment'] = df_fingpt['sentiment'].astype(str).str.lower().str.strip()
    df_fingpt['sentiment'] = df_fingpt['sentiment'].map(label_mapping)

    # Drop rows whose labels did not map to one of the 3 standard classes
    df_fingpt = df_fingpt.dropna(subset=['sentiment'])

    # Add source tracking
    df_fingpt['source'] = 'fingpt_news'

    return df_fingpt[['text', 'sentiment', 'source']]

In [24]:
def load_and_prep_financial_classification():
    """
    Loads the nickmuchi/financial-classification dataset (financial news headlines).
    Already provides clean 3-class sentiment annotations.
    """
    print("Loading Financial Classification Dataset (News Headlines)...")
    dataset = load_dataset("nickmuchi/financial-classification")

    # Combine all available splits
    frames = [pd.DataFrame(dataset[split]) for split in dataset.keys()]
    df_fc = pd.concat(frames, ignore_index=True)

    # Rename columns to our standard schema
    col_mapping = {}
    for col in df_fc.columns:
        if col.lower() in ['sentence', 'text', 'headline']:
            col_mapping[col] = 'text'
        elif col.lower() in ['sentiment', 'label', 'labels']:
            col_mapping[col] = 'sentiment'
    df_fc = df_fc.rename(columns=col_mapping)

    # Map numeric labels if necessary
    if pd.api.types.is_numeric_dtype(df_fc['sentiment']):
        label_mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}
        df_fc['sentiment'] = df_fc['sentiment'].map(label_mapping)
    else:
        df_fc['sentiment'] = df_fc['sentiment'].astype(str).str.lower().str.strip()

    # Keep only the 3 valid classes
    df_fc = df_fc[df_fc['sentiment'].isin(['negative', 'neutral', 'positive'])]

    # Add source tracking
    df_fc['source'] = 'financial_classification'

    return df_fc[['text', 'sentiment', 'source']]

### Merging the Expanded Sources

We now extend the existing `master_dataset` with the two new sources, applying the same cleaning logic (deduplication, NA removal, shuffling) used previously to ensure consistency.

In [25]:
def expand_master_dataset(df_base):
    """
    Extends an existing master dataset with FinGPT and Financial Classification sources.
    Applies the same cleaning pipeline (dedup, dropna, shuffle).
    """
    # 1. Load the new sources
    df_fingpt = load_and_prep_fingpt()
    df_fc = load_and_prep_financial_classification()

    # 2. Merge with the existing master
    print("Merging expanded sources with base master dataset...")
    df_expanded = pd.concat([df_base, df_fingpt, df_fc], ignore_index=True)

    # 3. Clean
    print("Cleaning expanded data...")
    df_expanded['text'] = df_expanded['text'].astype(str).str.strip()
    df_expanded = df_expanded[df_expanded['text'].str.len() > 0]
    df_expanded = df_expanded.drop_duplicates(subset=['text'])
    df_expanded = df_expanded.dropna()
    df_expanded = df_expanded.sample(frac=1, random_state=42).reset_index(drop=True)

    # 4. Summary
    print("\n--- Expanded Dataset Statistics ---")
    print(f"Total samples: {len(df_expanded)}")
    print("\nBreakdown by Source:")
    print(df_expanded['source'].value_counts())
    print("\nBreakdown by Sentiment:")
    print(df_expanded['sentiment'].value_counts())

    return df_expanded


expanded_dataset = expand_master_dataset(master_dataset)

Loading FinGPT Sentiment Dataset (Financial News)...
Loading Financial Classification Dataset (News Headlines)...
Merging expanded sources with base master dataset...
Cleaning expanded data...

--- Expanded Dataset Statistics ---
Total samples: 33298

Breakdown by Source:
source
fingpt_news                 20616
twitter                      6544
news                         4199
auditor                      1350
nickmuchi                     317
fiqa                          231
financial_classification       41
Name: count, dtype: int64

Breakdown by Sentiment:
sentiment
positive    13506
neutral     11891
negative     7901
Name: count, dtype: int64


### Balancing the Dataset for rsLoRA Fine-Tuning

RsLoRA (Rank-Stabilized LoRA) scales LoRA updates by `α / √r` instead of `α / r`, which **amplifies the effective gradient magnitude** of each adapter update. This makes the training dynamics more sensitive to class imbalance: an over-represented class can dominate the loss surface and push the rank-stabilized adapters toward a degenerate solution that simply predicts the majority class.

However, fully flattening the distribution (1:1) is also suboptimal:
- It discards real prior information (neutral *is* genuinely more common in financial text).
- It throws away large amounts of training signal, which hurts rsLoRA — a method that benefits from more samples to stabilize the higher-rank updates.

**Strategy used here — controlled imbalance with ratio 1.25:**
- Compute the minority-class size `m`.
- Cap every class at `1.25 × m` samples (the dominant class can be at most 25% larger than the minority).
- **Stratify** the downsampling by `source` so each class keeps a proportional mix of news, tweets, FinGPT, FiQA, and headlines — preserving domain diversity that rsLoRA needs to generalize.
- This preserves a mild natural prior toward `neutral` while keeping gradient signal balanced enough that rsLoRA's amplified updates do not collapse the model.

In [ ]:
import numpy as np
import pandas as pd

def balance_dataset(df, ratio=1.25, random_state=42):
    """
    Balance the dataset by capping each class at `ratio` * size_of_minority_class.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain a 'sentiment' column with values: negative, neutral, positive.
    ratio : float
        Max size of any class relative to the minority class.
        - ratio=1.0  -> perfectly balanced (loses more data)
        - ratio=1.25 -> mild imbalance, keeps more data (recommended)
        - ratio=1.5  -> slight imbalance, keeps even more
    random_state : int
        For reproducibility.

    Returns
    -------
    pd.DataFrame  (shuffled, index reset)
    """
    counts = df['sentiment'].value_counts()
    min_count = counts.min()
    cap = int(min_count * ratio)

    print(f"Minority class size: {min_count}")
    print(f"Cap per class (ratio={ratio}): {cap}")

    balanced_parts = []
    for label, group in df.groupby('sentiment'):
        if len(group) > cap:
            # Stratify by source to preserve source diversity within each class
            group_sampled = (
                group.groupby('source', group_keys=False)
                     .apply(lambda g: g.sample(
                         n=max(1, int(round(len(g) * cap / len(group)))),
                         random_state=random_state
                     ))
            )
            # Adjust if rounding produced more/less than cap
            if len(group_sampled) > cap:
                group_sampled = group_sampled.sample(n=cap, random_state=random_state)
            balanced_parts.append(group_sampled)
        else:
            balanced_parts.append(group)

    df_bal = pd.concat(balanced_parts, ignore_index=True)
    df_bal = df_bal.sample(frac=1, random_state=random_state).reset_index(drop=True)

    print("\n--- Balanced Dataset Statistics ---")
    print(f"Total samples: {len(df_bal)}")
    print("\nBreakdown by Sentiment:")
    print(df_bal['sentiment'].value_counts())
    print("\nBreakdown by Source:")
    print(df_bal['source'].value_counts())
    print("\nSentiment x Source matrix:")
    print(pd.crosstab(df_bal['sentiment'], df_bal['source']))

    return df_bal

balanced_dataset = balance_dataset(expanded_dataset, ratio=1.25)
balanced_dataset.to_csv("../data/sentiment_dataset_expanded.csv", index=False)

Minority class size: 7901
Cap per class (ratio=1.25): 9876

--- Balanced Dataset Statistics ---
Total samples: 27653

Breakdown by Sentiment:
sentiment
positive    9876
neutral     9876
negative    7901
Name: count, dtype: int64

Breakdown by Source:
source
fingpt_news                 17136
twitter                      5500
news                         3427
auditor                      1099
nickmuchi                     266
fiqa                          190
financial_classification       35
Name: count, dtype: int64

Sentiment x Source matrix:
source     auditor  financial_classification  fingpt_news  fiqa  news  \
sentiment                                                               
negative       117                         7         5315    72   509   
neutral        672                        28         5290    13  1836   
positive       310                         0         6531   105  1082   

source     nickmuchi  twitter  
sentiment                      
negative          92

C:\Users\david.suarez\AppData\Local\Temp\ipykernel_11168\2416857570.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(
C:\Users\david.suarez\AppData\Local\Temp\ipykernel_11168\2416857570.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(
